# Power Outages Analysis

**Name(s)**: Matthew Feng, Yu-Ming (Kenny) Wu

**Website Link**: https://mattf-owl.github.io/proj04_page/

## Step 1: Introduction

In this data science project, we are given a dataset about the major power outage caused by severe weather conditions in the US, from January 2000 to July 2006. The server outages (or major outages) is refers to  those that impacted least 50,000 customers or caused an unplanned firm load loss of atleast 300 MW. This defined by the Department of Energy. People can access to the original data at https://engineering.purdue.edu/LASCI/research-data/outages, which is published by Purdue University, Laboratory for Advancing Sustainable Critical Infrastructure.

The dataset includes the basic information (time and location) as well as the impacts regarding the power outages. Such as the general information - time and geographic areas, regional climate information, outage events information, cause of the event, extent of outages, regional electricity consumption information, regional economic characteristics, and regional land-use characterics. 

Our goal is to investigate further meaning in the data through statistical tests and eventually come up with a prediction model to estimate the duration of a power outage by information that can be gained when a power outage occurs. We are trying to figure out a research problem: which outage characteristics have the greatest impact on restoration time? To solveing this problem, we want to help people, especially policy makers, to figure out what can they do to reduce power outage duration, improve their living quality.

The data set originally have 1534 rows and 55 columns, which corresponded to 1534 outage events and 55 features for each outage event. There are few of these features listed below, which related with our analysis.

In [228]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
)
import seaborn as sns


pd.options.plotting.backend = "plotly"

# from dsc80_utils import * # Feel free to uncomment and use this.

In [229]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 500)
pd.set_option("display.max_colwidth", 200)

Since the raw data is in excel form, thus we removed 5 extra rows at the top and 1 extra column at the right side, moved the features name at the top of the dataframe

In [230]:
df = pd.read_excel(Path("data", "outage.xlsx"))
df.head(10)

,Major power outage events in the continental U.S.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,Time period: January 2000 - July 2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Regions affected: Outages reported in this data file affected a single U.S. state at the time of occurrence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
5,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,numeric,NaN,"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),NaN,NaN,NaN,mins,Megawatt,NaN,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,%,%,%,NaN,NaN,NaN,NaN,%,%,%,USD,USD,fraction,%,USD,USD,%,%,NaN,%,%,persons per square mile,persons per square mile,persons per square mile,%,%,%,%,%
6,NaN,1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00,severe weather,NaN,NaN,3060,NaN,70000,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.549073,32.225029,32.202431,2308736,276286,10673,2595696,88.944776,10.644005,0.411181,51268,47586,1.077376,1.6,4802,274182,1.751391,2.2,5348119,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
7,NaN,2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00,intentional attack,vandalism,NaN,1,NaN,NaN,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.032487,34.210389,35.727564,2345860,284978,9898,2640737,88.833534,10.791609,0.37482,53499,49091,1.089792,1.9,5226,291955,1.790002,2.2,5457125,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
8,NaN,3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:

In [231]:
df = df.iloc[4:, 1:].reset_index(drop=True)
df.head()

,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,numeric,NaN,"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),NaN,NaN,NaN,mins,Megawatt,NaN,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,%,%,%,NaN,NaN,NaN,NaN,%,%,%,USD,USD,fraction,%,USD,USD,%,%,NaN,%,%,persons per square mile,persons per square mile,persons per square mile,%,%,%,%,%
2,1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00,severe weather,NaN,NaN,3060,NaN,70000,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.549073,32.225029,32.202431,2308736,276286,10673,2595696,88.944776,10.644005,0.411181,51268,47586,1.077376,1.6,4802,274182,1.751391,2.2,5348119,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00,intentional attack,vandalism,NaN,1,NaN,NaN,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.032487,34.210389,35.727564,2345860,284978,9898,2640737,88.833534,10.791609,0.37482,53499,49091,1.089792,1.9,5226,291955,1.790002,2.2,5457125,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:00,severe weather,heavy wind,NaN,3000,NaN,70000,10.87,8.19,6.07,8.15,1467293,1801683,1951295,5222116,28.097672,34.501015,37.365983,2300291,276463,10150,2586905,88.920583,10.687018,0.392361,50447,47287,1.066826,2.7,4571,267895,1.706266,2.1,5310903,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


After remove these extra rows and column, the top two rows are feature names and the unit or description with corresponded feature names. To format this, I connection symbols from `.` to underscore `_` for giving a better visualizaiton. Also, combined the unites and the coresponding features together with a single space. Such as: `RES.PERCEN` -> `Res_percen (%)`, `RES.PRICE` -> `Res_Price (cents/kWh)` etc.


In [232]:
mapping_dict = {key: "" for key in df.iloc[1].unique()}
mapping_dict["cents / kilowatt-hour"] = "(cents / kWh)"
mapping_dict["USD"] = "($)"
mapping_dict["persons per square mile"] = "(people / sq mi)"
mapping_dict["%"] = "(%)"
df.iloc[0] = df.iloc[0].str.replace(".", "_").str.capitalize()
df.iloc[1] = df.iloc[1].map(mapping_dict)
df.iloc[0] = df.iloc[0].astype(str) + " " + df.iloc[1].astype(str)
df.columns = df.iloc[0].str.strip()
df = df.iloc[2:].set_index("Obs", drop=True)
df = df.rename(columns={"U_s__state": "US_state"})
df.head()

,Year,Month,US_state,Postal_code,Nerc_region,Climate_region,Anomaly_level,Climate_category,Outage_start_date,Outage_start_time,Outage_restoration_date,Outage_restoration_time,Cause_category,Cause_category_detail,Hurricane_names,Outage_duration,Demand_loss_mw,Customers_affected,Res_price (cents / kWh),Com_price (cents / kWh),Ind_price (cents / kWh),Total_price (cents / kWh),Res_sales,Com_sales,Ind_sales,Total_sales,Res_percen (%),Com_percen (%),Ind_percen (%),Res_customers,Com_customers,Ind_customers,Total_customers,Res_cust_pct (%),Com_cust_pct (%),Ind_cust_pct (%),Pc_realgsp_state ($),Pc_realgsp_usa ($),Pc_realgsp_rel,Pc_realgsp_change (%),Util_realgsp ($),Total_realgsp ($),Util_contri (%),Pi_util_ofusa (%),Population,Poppct_urban (%),Poppct_uc (%),Popden_urban (people / sq mi),Popden_uc (people / sq mi),Popden_rural (people / sq mi),Areapct_urban (%),Areapct_uc (%),Pct_land (%),Pct_water_tot (%),Pct_water_inland (%)
Obs,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00,severe weather,NaN,NaN,3060,NaN,70000,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.549073,32.225029,32.202431,2308736,276286,10673,2595696,88.944776,10.644005,0.411181,51268,47586,1.077376,1.6,4802,274182,1.751391,2.2,5348119,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00,intentional attack,vandalism,NaN,1,NaN,NaN,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.032487,34.210389,35.727564,2345860,284978,9898,2640737,88.833534,10.791609,0.37482,53499,49091,1.089792,1.9,5226,291955,1.790002,2.2,5457125,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:00,severe weather,heavy wind,NaN,3000,NaN,70000,10.87,8.19,6.07,8.15,1467293,1801683,1951295,5222116,28.097672,34.501015,37.365983,2300291,276463,10150,2586905,88.920583,10.687018,0.392361,50447,47287,1.066826,2.7,4571,267895,1.706266,2.1,5310903,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,2012,6,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19 00:00:00,04:30:00,2012-06-20 00:00:00,23:00:00,severe weather,thunderstorm,NaN,2550,NaN,68200,11.79,9.25,6.71,9.19,1851519,1941174,1993026,5787064,31.994099,33.54333,34.439329,2317336,278466,11010,2606813,88.895368,10.682239,0.422355,51598,48156,1.071476,0.6,5364,277627,1.932089,2.2,5380443,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
5,2015,7,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18 00:00:00,02:00:00,2015-07-19 00:00:00,07:00:00,severe weather,NaN,NaN,1740,250,250000,13.07,10.16,7.74,10.43,2028875,2161612,1777937,5970339,33.982576,36.20585,29.779498,2374674,289044,9812,2673531,88.821637,10.81132,0.367005,54431,49844,1.092027,1.7,4873,292023,1.668704,2.2,5489594,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


## Step 2: Data Cleaning and Exploratory Data Analysis
### Univariate Analysis 

Next, I select few relevant features that used for my analysis. They are: `US_state`, `Year`, `Pc_realgsp_state ($)`, `Outage_start_date`, `Outage_start_time`, `Outage_restoration_date`, `Outage_restoration_time`, `Total_price (cents / kWh)`, `Cause_category`, `Demand_loss_mw`, `Customers_affected`, `Total_sales`.

To get the outage restoration duration, we combined `Outage_start_date`, `Outage_start_time`, `Outage_restoration_date`, `Outage_restoration_time` together. First, we transform all these features into "datetime" data strucutre, then use `Outage_restoration_date` and `Outage_restoration_time` minus the `Outage_start_date` and `Outage_start_time` to get the restoration duration for each outage, transform the durating into the unit of hour. Lastly, drop columns regarding outage/restoration time  - `Outage_start_date`, `Outage_start_time`, `Outage_restoration_date`, `Outage_restoration_time`.

Notice there are `0` exist in `Outage_duration` which we calculated, which means there is 0 hour of outage happend, seems like a missing value. Thus we transform "0"s in the `Outage_duration` into `np.nan`.

In [233]:
cleaned_df = df[
    [
        "US_state",
        "Year",
        "Month",
        "Pc_realgsp_state ($)",
        "Outage_start_date",
        "Outage_start_time",
        "Outage_restoration_date",
        "Outage_restoration_time",
        "Total_price (cents / kWh)",
        "Cause_category",
        "Demand_loss_mw",
        "Customers_affected",
        "Total_sales",
    ]
].copy()

cleaned_df["Outage_start_date"] = pd.to_datetime(
    cleaned_df["Outage_start_date"], errors="coerce"
)

cleaned_df["Outage_restoration_date"] = pd.to_datetime(
    cleaned_df["Outage_restoration_date"], errors="coerce"
)


cleaned_df["Outage_start_time"] = pd.to_datetime(
    cleaned_df["Outage_start_time"], format="%H:%M:%S", errors="coerce"
).dt.time

cleaned_df["Outage_restoration_time"] = pd.to_datetime(
    cleaned_df["Outage_restoration_time"], format="%H:%M:%S", errors="coerce"
).dt.time

start = cleaned_df["Outage_start_date"] + pd.to_timedelta(
    cleaned_df["Outage_start_time"].astype(str)
)

end = cleaned_df["Outage_restoration_date"] + pd.to_timedelta(
    cleaned_df["Outage_restoration_time"].astype(str)
)

cleaned_df["Outage_duration"] = (end - start).dt.total_seconds() / 3600

cleaned_df.loc[cleaned_df["Outage_duration"] < 0, "Outage_duration"] = np.nan


cleaned_df = cleaned_df.drop(
    columns=[
        "Outage_start_date",
        "Outage_start_time",
        "Outage_restoration_date",
        "Outage_restoration_time",
    ]
)
# print(cleaned_df.head().to_markdown())
cleaned_df.head()

,US_state,Year,Month,Pc_realgsp_state ($),Total_price (cents / kWh),Cause_category,Demand_loss_mw,Customers_affected,Total_sales,Outage_duration
Obs,,,,,,,,,,
1,Minnesota,2011,7,51268,9.28,severe weather,NaN,70000,6562520,51.000000
2,Minnesota,2014,5,53499,9.28,intentional attack,NaN,NaN,5284231,0.016667
3,Minnesota,2010,10,50447,8.15,severe weather,NaN,70000,5222116,50.000000
4,Minnesota,2012,6,51598,9.19,severe weather,NaN,68200,5787064,42.500000
5,Minnesota,2015,7,54431,10.43,severe weather,250,250000,5970339,29.000000


For exploring the data set, we used univariate analysis to visulize the distribution or the other information of a single feature.

First, we draw the histogram on feature `Outage_duration`:

In [234]:
fig = px.histogram(
    cleaned_df,
    x="Outage_duration",
    nbins=50,
    title="Distribution of Outage Duration (hours)",
    labels={"Outage_duration": "Outage Duration (hours)"},
)
fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/dist_of_outage_duration.html"
# )

Through the histogram graph above, we can see that the outage duration have a right skew shape, shows most outage duration being fixed really fast.

Second, we can visualize the `Pc_realgsp_state` by histogram, to see the distribution of the real GSP per state, in dollar:

In [235]:
fig = px.histogram(
    cleaned_df,
    x="Pc_realgsp_state ($)",
    nbins=20,
    title="Distribution of Real GSP per State ($)",
)
fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/dist_of_real_GSP_per_state.html"
# )

Through the histogram above, we can see most US states have averagely 50k of real gsp at most time from January 2000 to July 2016. Only few states have 160k of real gsp at some time between January 2000 to July 2016.

Last, we can visualize the number of outage happened in each state of US, from January 2000 to July 2016:

In [236]:
import folium
import pandas as pd

state_counts = (
    cleaned_df.groupby("US_state").size().reset_index(name="num_outages")
)

url = "https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json"

m1 = folium.Map(location=[37.8, -96], zoom_start=4)

folium.Choropleth(
    geo_data=url,
    data=state_counts,
    columns=["US_state", "num_outages"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Number of Power Outages",
).add_to(m1)

# m1.save(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/number_of_power_outage.html"
# )
m1

Through the map above, we can see that the most number of outages are happened in California, and the second most number of outages are happened in Texas.

## Bivariate Analysis

Also we used bivariate analysis to visulize the distribution or the other information of two features. To help me understand some relation between pairs of features.

First, we want to visualize the relation between the causation of outages and the median of outage duration by bar graph. we chose to use median of outage duration because of the distribution of outage duration is skewed, using median can better explain the distribution of outage duration.


In [237]:
median_df = cleaned_df.groupby("Cause_category", as_index=False)[
    "Outage_duration"
].median()

fig = px.bar(
    median_df,
    x="Cause_category",
    y="Outage_duration",
    title="Median Outage Duration by Cause Category",
    labels={
        "Cause_category": "Cause Category",
        "Outage_duration": "Median Outage Duration (hours)",
    },
    color="Cause_category",
    color_discrete_sequence=px.colors.qualitative.Set3,
)

fig.update_layout(showlegend=False)

fig.show()
fig.write_html(
    "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/median_outage_duration_by_cause.html"
)

Through the bar graph, we can see that most outages is caused by the fuel supply emergency. The second main cause of power outage is severe weather.

Second, we can visualize the average outage duration in each state of US, from January 2000 to July 2016:

In [238]:
state_duration = (
    cleaned_df.groupby("US_state")["Outage_duration"].mean().reset_index()
)

m2 = folium.Map(location=[37.8, -96], zoom_start=4)

folium.Choropleth(
    geo_data=url,
    data=state_duration,
    columns=["US_state", "Outage_duration"],
    key_on="feature.properties.name",
    fill_color="PuBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Average Restoration Time (hours)",
).add_to(m2)

# m2.save(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/avg_outage_duration_map.html"
# )
m2

Through the map above, we can see that Wisconsin and West Virginia take longest average time to restore power to the outage area.

## Grouping and Aggregates 

We create a pivot-table to see the median outage duration by each cause over years. Using median value can better represent the shape and the distribution of the skewed data. The columns are the corresponded years, the rows are corresponded causation of power outages. The pivot-table is presented as below:

In [239]:
# Aggregate analysis
pivot = cleaned_df.pivot_table(
    values="Outage_duration",
    index="Cause_category",
    columns="Year",
    aggfunc="median",
)
# print(pivot.to_markdown())
pivot

Year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016
Cause_category,,,,,,,,,,,,,,,,,
equipment failure,NaN,8.233333,NaN,2.591667,3.683333,2.483333,2.650000,3.433333,4.483333,15.450000,4.916667,0.066667,1.750000,3.258333,NaN,NaN,NaN
fuel supply emergency,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.933333,396.000000,109.500000,168.000000,78.750000,60.000000,43.716667,93.575000,4.250000,568.366667
intentional attack,NaN,NaN,NaN,22.158333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.166667,0.250000,1.000000,0.150000,1.000000,1.300000
islanding,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.783333,3.725000,4.841667,2.033333,0.883333,0.333333,0.533333,0.933333,0.583333,3.716667
public appeal,NaN,2.333333,NaN,25.800000,72.000000,NaN,5.950000,40.508333,100.958333,5.000000,8.533333,10.000000,5.000000,NaN,6.700000,5.558333,NaN
severe weather,41.500000,169.000000,86.000000,62.233333,34.500000,73.500000,45.541667,25.500000,53.000000,45.166667,48.875000,38.316667,41.783333,31.000000,16.666667,32.500000,23.416667
system operability disruption,6.258333,4.016667,3.833333,44.900000,1.583333,4.116667,3.950000,6.916667,3.241667,1.775000,3.316667,4.866667,0.216667,2.016667,0.866667,1.083333,3.700000


In this pivot table, the "nan" value shows there is not power outage happened in the corresponded year with the corrponded causations. I notice that on row "servere weather" and "system operability disruption" there are power outages happened every year, showing these two causation is more common than other 6 power outage causations.

Also, in the row of "fuel supply emergency", the duration of outage restoration is averagely longer then all other 7 power outage causations based on the corresponded years, showing "fuel supply emergency" is a severe power outage causation.

## Step 3: Assessment of Missingness

In the cleaned dataframe, there are columns with missing values: Total_price (cents / kWh), Demand_loss_mw, Customers_affected, and Total_sales. The following series shows the missingness proportion of each column. 

In [240]:
# missing values
cleaned_df.isna().sum() / cleaned_df.shape[0]

0
US_state                     0.000000
Year                         0.000000
Month                        0.005867
Pc_realgsp_state ($)         0.000000
Total_price (cents / kWh)    0.014342
Cause_category               0.000000
Demand_loss_mw               0.459583
Customers_affected           0.288787
Total_sales                  0.014342
Outage_duration              0.037810
dtype: float64

## NMAR Analysis
Among these columns, "Demand_loss_mw" and "Customers_affected" are likely to be MNAR, because the missingness may depend on the companies which measure the scale of impact of the power outages. When the outage is severe, it may be difficult to collect data over a wide range. On the other hand, when the impact of the outage is negligible, the companies may just ignore and not to report it. It is also possible that certain companies are not reporting, but we still cannot conclude the exact reason why some data is missing. 

## MAR Analysis 
Now, we want to find out the missingness dependency of "Total_price (cents / kWh)" and "Total_sales". By examining the dataframe, we found out that both columns have missing value in the year of 2016, so we decided to  perform some permutation test to confirm this dependency. The following the code helps us to evaluate the dependency of the missingness

In [241]:
cat_cols = ["US_state", "Year", "Cause_category"]
quan_cols = [
    "Pc_realgsp_state ($)",
    "Total_price (cents / kWh)",
    "Demand_loss_mw",
    "Customers_affected",
    "Total_sales",
    "Outage_duration",
]


def missingness_perm_test(df, miss_col, dep_col, test_stat, n_perm=1000):

    # missingness indicator
    missing = df[miss_col].isna()

    # observed statistic
    obs = test_stat(df, missing, dep_col)

    stats = []

    for _ in range(n_perm):

        shuffled = np.random.permutation(missing)

        stat = test_stat(df, shuffled, dep_col)

        stats.append(stat)

    p_value = np.mean(np.array(stats) >= obs)

    return p_value


def tvd_stat(df, missing, dep_col):

    dist = (
        pd.DataFrame({"dep": df[dep_col], "missing": missing})
        .groupby("dep")["missing"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )

    overall = pd.Series(missing).value_counts(normalize=True)

    tvd = 0.5 * np.abs(dist.sub(overall, axis=1)).sum(axis=1).mean()

    return tvd


def diff_mean_stat(df, missing, dep_col):

    return abs(
        df.loc[missing, dep_col].mean() - df.loc[~missing, dep_col].mean()
    )

In [242]:
def helper_missing(test_col):
    for col in cleaned_df.columns:

        if col == test_col or "missing" in col:
            continue

        if col in cat_cols:
            print(
                col,
                missingness_perm_test(
                    cleaned_df,
                    test_col,
                    col,
                    tvd_stat,
                ),
            )

        elif col in quan_cols:
            print(
                col,
                missingness_perm_test(
                    cleaned_df,
                    test_col,
                    col,
                    diff_mean_stat,
                ),
            )

In [243]:
helper_missing("Total_price (cents / kWh)")

US_state 0.0
Year 0.0
Pc_realgsp_state ($) 0.347
Cause_category 0.047
Demand_loss_mw 0.623
Customers_affected 0.25
Total_sales 0.0
Outage_duration 0.106


In [244]:
helper_missing("Demand_loss_mw")

US_state 0.0
Year 0.0
Pc_realgsp_state ($) 0.204
Total_price (cents / kWh) 0.475
Cause_category 0.0
Customers_affected 0.116
Total_sales 0.003
Outage_duration 0.7


In [245]:
helper_missing("Customers_affected")

US_state 0.0
Year 0.0
Pc_realgsp_state ($) 0.288
Total_price (cents / kWh) 0.476
Cause_category 0.0
Demand_loss_mw 0.0
Total_sales 0.452
Outage_duration 0.028


In [246]:
helper_missing("Total_sales")

US_state 0.0
Year 0.0
Pc_realgsp_state ($) 0.345
Total_price (cents / kWh) 0.0
Cause_category 0.048
Demand_loss_mw 0.643
Customers_affected 0.22
Outage_duration 0.112


In [247]:
helper_missing("Outage_duration")

US_state 0.001
Year 0.0
Pc_realgsp_state ($) 0.966
Total_price (cents / kWh) 0.768
Cause_category 0.0
Demand_loss_mw 0.518
Customers_affected 0.661
Total_sales 0.515


The following figure shows the frequency of outage over year conditioned on whether "Total_price (cents / kWh)" is missing. 

In [248]:
df_plot = cleaned_df.copy()

df_plot["price_missing"] = df_plot["Total_price (cents / kWh)"].isna()

fig = px.histogram(
    df_plot,
    x="Year",
    color="price_missing",
    barmode="group",
    category_orders={"price_missing": [False, True]},
    labels={"price_missing": "Price Missing"},
    title="Missing vs Observed Total_price (cents / kWh) by Year",
)

fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/total_price_missing.html"
# )

Then we set up a hypothesis test to find the missing mechanism of feature `Total_price (cents / kWh)` by using permutation test. We will set the significance level into 0.05.

**Null Hypothesis:**  The distribution of `Year` is same when `Total_price (cents / kWh)` is missing vs not missing.

**Alternate Hypothesis:** The distribution of `Year` is different when `Total_price (cents / kWh)` is missing vs not missing.

**Test Statistic:** TVD - Total Variation Distance

The following figure shows the result the permutation test:

In [249]:
missing = cleaned_df["Total_price (cents / kWh)"].isna()

obs_tvd = tvd_stat(cleaned_df, missing, "Year")

perm_stats = []

for _ in range(1000):
    shuffled = pd.Series(np.random.permutation(missing), index=missing.index)
    perm_stats.append(tvd_stat(cleaned_df, shuffled, "Year"))

fig = px.histogram(
    perm_stats,
    nbins=40,
    title="Permutation Distribution of TVD Statistic",
    labels={"value": "TVD"},
)

fig.add_vline(
    x=obs_tvd,
    line_color="red",
    line_width=3,
    annotation_text="Observed TVD",
    annotation_position="top right",
)

fig.add_vline(
    x=np.percentile(perm_stats, 95),
    line_color="black",
    line_width=3,
    annotation_text="Significant value = 0.05",
    annotation_position="top right",
)

fig.update_layout(showlegend=False)

fig.show()

# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/total_price_missing_perm.html"
# )
p_value = (perm_stats >= obs_tvd).mean()
print(f"p_value: {p_value}")

p_value: 0.0


Through the graph above, we can see the observed TVD of feature `Total_price (cents / kWh)` missingness has a p-value 0.0. The result show we reject the null hypothesis. This result means the feature `Total_price (cents / kWh)` missing value is dependent on the feature `Year`, showing `Total_price (cents / kWh)` is likely to be MAR.

Similarly, The following figure shows the frequency of outage over year conditioned on whether "Total_sales" is missing. 

In [250]:
df_plot["sales_missing"] = df_plot["Total_sales"].isna()

fig = px.histogram(
    df_plot,
    x="Customers_affected",
    color="sales_missing",
    barmode="group",
    category_orders={"sales_missing": [False, True]},
    labels={"sales_missing": "Sales Missing"},
    title="Missing vs Observed Total_sales by Customers_affected",
)

fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/total_sales_missing.html"
# )

Then we set up a hypothesis test to find the missing mechanism of feature `Total_sales` by using permutation test. We will set the significance level into 0.05.

**Null Hypothesis:**  The distribution of `Customers_affected` is same when `Total_sales` is missing vs not missing.

**Alternate Hypothesis:** The distribution of `Customers_affected` is different when `Total_sales` is missing vs not missing.

**Test Statistic:** Absolute mean difference 

The following figure shows the result the permutation test. The test statistic is the total variance distance between two groups because "Year" is a categorical feature. 

In [251]:
missing = cleaned_df["Total_sales"].isna()

obs_stat = diff_mean_stat(cleaned_df, missing, "Customers_affected")

perm_stats = []

for _ in range(1000):
    shuffled = pd.Series(np.random.permutation(missing), index=missing.index)
    perm_stats.append(
        diff_mean_stat(cleaned_df, shuffled, "Customers_affected")
    )

fig = px.histogram(
    perm_stats,
    nbins=40,
    title="Permutation Distribution of Absolute Mean Difference",
    labels={"value": "Absolute Mean Difference"},
)

fig.add_vline(
    x=obs_stat,
    line_color="red",
    line_width=3,
    annotation_text="Observed statistics",
    annotation_position="top right",
)

fig.add_vline(
    x=np.percentile(perm_stats, 95),
    line_color="black",
    line_width=3,
    annotation_text="Significant value = 0.05",
    annotation_position="top right",
)

fig.update_layout(showlegend=False)

fig.show()

# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/total_sales_missing_perm.html"
# )
p_value = (perm_stats >= obs_stat).mean()
print(f"p_value: {p_value}")

p_value: 0.231


Through the graph above, we can see the observed TVD of feature `Total_sales (cents / kWh)` missingness has a p-value 0.0. The result show we reject the null hypothesis. This result means the feature `Total_sales (cents / kWh)` missing value is dependent on the feature `Year`, showing `Total_sales (cents / kWh)` is likely to be MAR.

Lastly, we set up a hypothesis test to find the missing mechanism of feature `Outage_duration` by using permutation test. We will set the significance level into 0.05.

In [252]:
# outage duration
df_plot = cleaned_df.copy()

df_plot["outage_duration_missing"] = df_plot["Outage_duration"].isna()

fig = px.histogram(
    df_plot,
    x="Cause_category",
    color="outage_duration_missing",
    barmode="group",
    histnorm="percent",  # <-- key line
    category_orders={"outage_duration_missing": [False, True]},
    labels={"outage_duration_missing": "Outage Duration Missing"},
    title="Percentage Missing vs Observed Outage Duration by Cause_category",
)

fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/outage_missing.html"
# )

**Null Hypothesis:**  The distribution of `Cause_category` is same when `Outage_Duration` is missing vs not missing.

**Alternate Hypothesis:** The distribution of `Cause_category` is different when `Outage_Duration` is missing vs not missing.

**Test Statistic:** TVD - Total Variation Distance

In [253]:
missing = cleaned_df["Outage_duration"].isna()

obs_tvd = tvd_stat(cleaned_df, missing, "Cause_category")

perm_stats = []

for _ in range(1000):
    shuffled = pd.Series(np.random.permutation(missing), index=missing.index)
    perm_stats.append(tvd_stat(cleaned_df, shuffled, "Cause_category"))

fig = px.histogram(
    perm_stats,
    nbins=40,
    title="Permutation Distribution of TVD Statistic",
    labels={"value": "TVD"},
)

fig.add_vline(
    x=obs_tvd,
    line_color="red",
    line_width=3,
    annotation_text="Observed TVD",
    annotation_position="top right",
)

fig.add_vline(
    x=np.percentile(perm_stats, 95),
    line_color="black",
    line_width=3,
    annotation_text="Significance level = 0.05",
    annotation_position="top right",
)

fig.update_layout(showlegend=False)

fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/outage_missing_perm.html"
# )

In [254]:
print(obs_tvd, perm_stats)

0.0467486098156178 [np.float64(0.01589598194440267), np.float64(0.01587618715518366), np.float64(0.028734809066250803), np.float64(0.020223979133810337), np.float64(0.015155308150392071), np.float64(0.017818237896757703), np.float64(0.012754262442787628), np.float64(0.024809749872416235), np.float64(0.005482191097048686), np.float64(0.01507693162562297), np.float64(0.010651774476405588), np.float64(0.012406017259495314), np.float64(0.01122205866650018), np.float64(0.010814200896099635), np.float64(0.01570213828716246), np.float64(0.015327641416256645), np.float64(0.013742187228147419), np.float64(0.012041959329959249), np.float64(0.02515805352555078), np.float64(0.02322107316895942), np.float64(0.01117926344397059), np.float64(0.01536307879431909), np.float64(0.01293209411967906), np.float64(0.01613232291026827), np.float64(0.025879959923008462), np.float64(0.014357480004364456), np.float64(0.01871224454757691), np.float64(0.027589847183396576), np.float64(0.015039107819506405), np.flo

Through the graph above, we can see the observed TVD of feature `Outage_Duration` missingness has a p-value 0.0. The result show we reject the null hypothesis. This result means the feature `Outage_Duration` missing value is dependent on the feature `Cause_category`, showing `Outage_Duration` is likely to be MAR.

## Handling Missingness
The following table shows how we handled the missingness. "Demand_loss_mw" and "Outage_duration" are conditioned on "Cause_category" because they have a closer relationship. On the other hand, "Customer_affected", "Total_Sales", and "Total_price (cents / kWh)" are conditioned on "US_State" because the values are correlated to a state's population and economic status. The choice of mean and median is determined by the distribution of the numerical values in each feature. Median is chosen when the distribution is skewed, vice versa. 

| Column | Method | 
|--------|--------|
| "Demand_loss_mw" | group mean inputation on "Cause_category" |
| "Customers_affected" | group mean inputation conditioned on "US_state" |
| "Total Sales" | group mean inputation on "US_state" |
| "Total_price (cents / kWh)" | group mean inputation conditioned on "US_state" |
| "Outage_duration" | group mean inputation conditioned on "Cause_category" |

In [255]:
# TODO
cleaned_df["Demand_loss_mw"] = cleaned_df.groupby("Cause_category")[
    "Demand_loss_mw"
].transform(lambda x: x.fillna(x.mean()))

# customers affected
cleaned_df["Customers_affected"] = cleaned_df.groupby("US_state")[
    "Customers_affected"
].transform(lambda x: x.fillna(x.mean()))

# total sales
cleaned_df["Total_sales"] = cleaned_df.groupby("US_state")[
    "Total_sales"
].transform(lambda x: x.fillna(x.mean()))

cleaned_df["Total_price (cents / kWh)"] = cleaned_df.groupby("US_state")[
    "Total_price (cents / kWh)"
].transform(lambda x: x.fillna(x.mean()))

# outage duration
cleaned_df["Outage_duration"] = cleaned_df.groupby("Cause_category")[
    "Outage_duration"
].transform(lambda x: x.fillna(x.median()))

cleaned_df = cleaned_df.dropna(
    subset=[
        "Demand_loss_mw",
        "Customers_affected",
        "Total_sales",
        "Total_price (cents / kWh)",
        "Outage_duration",
    ]
)

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/1247500671.py:4: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/1247500671.py:9: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/1247500671.py:14: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

## Step 4: Hypothesis Testing

We will be testing on whether richer/more developed states in the US have a different outage duration, meaning that the power restoration comes back faster in states with better economic status. After finding the average real GSP of all states (roughly 50,000), we decided to label each state by "above 50,000" and "below 50,000" (Since feature `Pc_realgsp_state ($)` have a roughly normal distribution, which we find the median and mean of this feature is rouphly 50,000, thus,  mean is an appropriate measure of center using for the permutation test).

**Null hypothesis:** The duration of the restoring time has no difference between the states with more than 50,000 real GSP and the states with less than 50,000 GSP.

**Alternative hypothesis:** There is a difference in the power outage duration between states with more than 50,000 real GSP and those with less than 50,000 GSP.

**Test statistics:** The absolute difference in the median of 2 groups. We chose median because the distribution of the power outage duration is right-skewed. The absolute value is for observing the difference between categorical groups. 

The following graph shows the distribution of the power outage duration between two groups. The label "True" indicates the power outages occured in states with a real GSP over 50,000. False just means the other way. 

In [256]:
df_hypo_h0 = cleaned_df[["Pc_realgsp_state ($)", "Outage_duration"]].copy()
df_hypo_h0["Pc_realgsp_state ($)"] = (
    df_hypo_h0["Pc_realgsp_state ($)"] >= 50000
)

fig = px.histogram(
    df_hypo_h0,
    x="Outage_duration",
    color="Pc_realgsp_state ($)",
    barmode="overlay",
    histnorm="percent",
)

fig.update_traces(opacity=0.6)
fig.show()

# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/outage_duration_with_labels.html"
# )

The following code does the permutation test:

In [257]:
group_medians = df_hypo_h0.groupby("Pc_realgsp_state ($)")[
    "Outage_duration"
].median()

test_stat = abs(group_medians.loc[True] - group_medians.loc[False])
arr = []

for _ in range(1000):
    shuffled = np.random.permutation(df_hypo_h0["Pc_realgsp_state ($)"])
    tmp = df_hypo_h0.copy()
    tmp["shuffle"] = shuffled
    tmp_medians = tmp.groupby("shuffle")["Outage_duration"].median()
    arr.append(abs(tmp_medians.loc[True] - tmp_medians.loc[False]))

arr = np.array(arr)
p_value = np.mean(arr >= test_stat)

p_value

np.float64(0.0)

The following figure visualizes the result of the permutation test, where the red axis represents our observed test statistic. The p-value of this test is 0.0, meaning that none of the iterations in the test has a greater test statistics than our observed test statistic (same as the figure). Therefore, we reject the null hypothesis and conclude that there is a difference in the distributions of outage duration between two labels of states. 

In [258]:
import plotly.graph_objects as go

fig = go.Figure()

# histogram of permutation statistics
fig.add_trace(
    go.Histogram(
        x=arr, nbinsx=40, name="Permutation distribution", opacity=0.75
    )
)

# vertical line for observed statistic
fig.add_vline(
    x=test_stat,
    line_color="red",
    line_width=3,
    annotation_text=f"Observed stat = {test_stat:.2f}",
    annotation_position="top",
)

fig.add_vline(
    x=np.percentile(arr, 95),
    line_color="black",
    line_width=3,
    annotation_text=f"Significance level 0.05",
    annotation_position="top",
)

fig.update_layout(
    title="Permutation Test Distribution",
    xaxis_title="Difference in Median Outage Duration",
    yaxis_title="Count",
    showlegend=False,
)

fig.show()

# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/perm_test_dist.html"
# )

## Step 5: Framing a Prediction Problem

Our model aims to estimate/predict the power outage duration by the selected features ("Pc_realgsp_state ($)", "Total_price (cents / kWh)", "Cause_category", "Demand_loss_mw", "Customers_affected", "Total_sales",). This is a typical regression problem, so the performance matrix chosen is root mean square error (RMSE). The lower RMSE, the closer distance between the predicted value and the actual value. 

## Step 6: Baseline Model

Our model is Multiple Linear Regression, using features  `Month`, `US_state`, `Pc_realgsp_state ($)`, `Total_price (cents / kWh)`, `Cause_category`, `Demand_loss_mw`, `Customers_affected`, `Total_sales` to build oure baseline model, which to predict the duration of power outage restoration. All these informations can help people to better estimate the time length of power outage restoration.

We have ordinal feature: `Month`, nominal feautures: `US_state`, `Cause_category`, and quantitative features: `Pc_realgsp_state ($)`, `Total_price (cents / kWh)`, `Demand_loss_mw`, `Customers_affected`, `Total_sales`. We believe these features are relate with your response variabe `Outage_duration`, such as different region may have different approach to restore the power outage, so we use the feature `US_state`; the state real Gross State Product (GSP) represent the financial strength of the state may affect the time for restore the power outage, so we use the feature `Pc_realgsp_state ($)`.

In the baseline model we use `OneHotEncoder` to transform the ordinal features and nominal features in to 0 or 1 distinct groups, and we drop the first column after feature bing one-hot encoded to prevent the collinearity. Encoding these nominal and ordinal features help use convert object-type features in to numerical type which can used for training the model.

The performance of this model is not so good, which we get a mean absolute error of 21.6324, a R squared score of 0.02806. The MAE shows the model predict the outage duration deviate from the actual outage duration, and R squared score indicates the model explain very small proportion of the variability of the actual outage duration.

### Data Cleaning for Modeling

Before modeling, we notice that there is a small number of unusually large  `Outage_duration` values may reduce the performance of the model, which these data are extreme and not representative of most cases. 

Thus, we seems these data as outliers, as defined by the IQR method, which defines outliers as values either below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR, where Q1 is the 25 percentile of the data, Q3 is the 75 percentile of the data, and IQR is range between Q1 and Q3 (Q3 - Q1).

We will use the data set without these `Outage_duration` outliers for both baseline modeling and the final modeling.

In [260]:
third_qrt = cleaned_df["Outage_duration"].quantile(0.75)
first_qrt = cleaned_df["Outage_duration"].quantile(0.25)
bound = 1.5 * (third_qrt - first_qrt)
no_outliers = cleaned_df[cleaned_df["Outage_duration"] <= third_qrt + bound]
no_outliers = no_outliers[no_outliers["Month"].notna()]

In [267]:
X = no_outliers[
    [
        "Month",
        "US_state",
        "Pc_realgsp_state ($)",
        "Total_price (cents / kWh)",
        "Cause_category",
        "Demand_loss_mw",
        "Customers_affected",
        "Total_sales",
    ]
]

y = no_outliers["Outage_duration"]


cat_cols = ["Month", "US_state", "Cause_category"]

num_cols = [
    "Pc_realgsp_state ($)",
    "Total_price (cents / kWh)",
    "Demand_loss_mw",
    "Customers_affected",
    "Total_sales",
]

preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    remainder="passthrough",
)


pl = make_pipeline(
    preprocessor,
    LinearRegression(),
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pl.fit(X_train, y_train)
pred = pl.predict(X_test)
mae = mean_absolute_error(y_test, pred)
mae, r2_score(y_test, pred)

(np.float64(21.63237137516751), 0.028061903505385732)

The model performance can be visualized below, which the scatter plot shows how predict value is far away from the actual value by the y=x reference line:

In [268]:
plot_df = pd.DataFrame({"Actual": y_test, "Predicted": pred})

fig = px.scatter(
    plot_df,
    x="Actual",
    y="Predicted",
    title="Predicted vs Actual Outage Duration",
    labels={
        "Actual": "Actual Outage Duration",
        "Predicted": "Predicted Outage Duration",
    },
)

max_val = max(plot_df["Actual"].max(), plot_df["Predicted"].max())

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val + 5,
    y1=max_val + 5,
    line=dict(color="red", dash="dash"),
)

fig.update_xaxes(range=[0, max_val + 5])
fig.update_yaxes(range=[0, max_val + 5])

fig.show()


fig.write_html(
    "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/baseline.html"
)

## Step 7: Final Model

To get a better estimate/prediction on the outage duration, we select new features, done some feature engineering, choose a new model, and use CV fold to train the model.

### Feature Selection
We use the pearson correlation to compared between feature `Outage_duration` with all other features, and get 3 more features: `Util_contri (%)`, `Pct_land (%)`, and `Pct_water_tot (%)` that have higher pearson correlations then all other features that not being selected in the baseline model. We included these new three features into our predictor data set.

In [266]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

df.dtypes

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/661342037.py:2: FutureWarning:

errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead



0
Year                                      int64
Month                                   float64
US_state                                 object
Postal_code                              object
Nerc_region                              object
Climate_region                           object
Anomaly_level                           float64
Climate_category                         object
Outage_start_date                datetime64[ns]
Outage_start_time                        object
Outage_restoration_date          datetime64[ns]
Outage_restoration_time                  object
Cause_category                           object
Cause_category_detail                    object
Hurricane_names                          object
Outage_duration                         float64
Demand_loss_mw                          float64
Customers_affected                      float64
Res_price (cents / kWh)                 float64
Com_price (cents / kWh)                 float64
Ind_price (cents / kWh)               

In [188]:
high_corr = (
    pd.concat(
        [
            df.select_dtypes(include=["int", "float"]),
            cleaned_df[["Outage_duration"]],
        ],
        axis=1,
    )
    .corr()
    .iloc[:, -1]
)

a = high_corr[high_corr.abs() >= 0.1].index.to_list()
a

['Year',
 'Outage_duration',
 'Customers_affected',
 'Util_contri (%)',
 'Pct_land (%)',
 'Pct_water_tot (%)',
 'Outage_duration']

In [269]:
no_outliers = no_outliers.merge(
    df[a[3:6]], left_index=True, right_index=True, how="left"
)
no_outliers.head()

,US_state,Year,Month,Pc_realgsp_state ($),Total_price (cents / kWh),Cause_category,Demand_loss_mw,Customers_affected,Total_sales,Outage_duration,Util_contri (%),Pct_land (%),Pct_water_tot (%)
Obs,,,,,,,,,,,,,
1,Minnesota,2011,7,51268,9.28,severe weather,618.661578,70000.000000,6562520.0,51.000000,1.751391,91.592666,8.407334
2,Minnesota,2014,5,53499,9.28,intentional attack,9.151351,124006.571429,5284231.0,0.016667,1.790002,91.592666,8.407334
3,Minnesota,2010,10,50447,8.15,severe weather,618.661578,70000.000000,5222116.0,50.000000,1.706266,91.592666,8.407334
4,Minnesota,2012,6,51598,9.19,severe weather,618.661578,68200.000000,5787064.0,42.500000,1.932089,91.592666,8.407334
5,Minnesota,2015,7,54431,10.43,severe weather,250.000000,250000.000000,5970339.0,29.000000,1.668704,91.592666,8.407334


### Feature Engineering
1. we mapping the feature `Month` into `Season`, from ordinal into nominal which each three month will be transformed in to their corresponded season, such as 12, 1, 2 will map to  "Winter". To do so, we expected using season instead of month having more explanatory power, which the power outage duration may affected by the seasonal climate. This feature transformation can also reduced the noise which reduced from 12 categories into 4 categories. We then include this transformer into our pipeline.

2. We put another transformer - `StandardScaler` - into our pipeline for quantitative data. Using `StandardScaler` can bring all quantitative data more centralized to 0 and keep the shpae of the feature distribution. This transformer can help model train more stably, reduce the number scale of the large=scale features.

In [275]:
X = no_outliers.drop(columns=["Outage_duration", "Year"])
y = no_outliers["Outage_duration"]


def month_season(s):
    return (
        s["Month"]
        .map(
            {
                1: "Winter",
                2: "Winter",
                12: "Winter",
                3: "Spring",
                4: "Spring",
                5: "Spring",
                6: "Summer",
                7: "Summer",
                8: "Summer",
                9: "Fall",
                10: "Fall",
                11: "Fall",
            }
        )
        .to_frame()
    )


num_cols = X.select_dtypes(exclude=["object"]).columns.to_list()

season_pipe = make_pipeline(
    FunctionTransformer(month_season), OneHotEncoder(handle_unknown="ignore")
)
preprocessor = make_column_transformer(
    (season_pipe, ["Month"]),
    (StandardScaler(), num_cols),
    (OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
)

### Model Selection
We changed our model from Multiple Linear Regression into Random Forest Regressor. This model can better capture the nonlinear relationships and be more robust to outliers and unusual patterns, which we expect will have better performance then Multiple Linear Regression. IN Random Forest Regressor, we choose to use hyperparameters: `n_estimators`, `max_depth`, `min_samples_leaf`.

In [279]:
pl = make_pipeline(
    preprocessor,
    RandomForestRegressor(random_state=42),
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    "randomforestregressor__n_estimators": np.arange(100, 500, 50),
    "randomforestregressor__max_depth": np.arange(5, 30, 5),
    "randomforestregressor__min_samples_leaf": np.arange(1, 10, 2),
}

grid = GridSearchCV(
    pl,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
    n_jobs=-1,
)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_
pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, pred)

mae, r2_score(y_test, pred)

/Users/wuyuming/miniforge3/envs/dsc80/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/wuyuming/miniforge3/envs/dsc80/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/wuyuming/miniforge3/envs/dsc80/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/wuyuming/miniforge3/envs/dsc80/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Use

(np.float64(15.302998753662918), 0.3639904323916331)

### Model Training
We used the `GridSearchCV` with `cv` of 5 and `scoring` of "neg_mean_absolute_error". The `GridSearchCV` find the best hyperparameters of:
    *max_depth: 10
    *min_samples_leaf: 1
    *n_estimators: 200

In [280]:
grid.best_params_

{'randomforestregressor__max_depth': np.int64(25),
 'randomforestregressor__min_samples_leaf': np.int64(3),
 'randomforestregressor__n_estimators': np.int64(350)}

In [281]:
results = pd.DataFrame(grid.cv_results_)
results["mae_val"] = -results["mean_test_score"]
results["mae_train"] = -results["mean_train_score"]
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_randomforestregressor__max_depth,param_randomforestregressor__min_samples_leaf,param_randomforestregressor__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score,mae_val,mae_train
0,0.268471,0.015490,0.004708,0.000232,5,1,100,"{'randomforestregressor__max_depth': 5, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 100}",-15.310899,-14.663847,-14.200810,-14.857934,-14.173658,-14.641429,0.426268,180,-12.355646,-12.441507,-12.567303,-12.460169,-12.725525,-12.510030,0.127070,14.641429,12.510030
1,0.343940,0.017521,0.009948,0.005823,5,1,150,"{'randomforestregressor__max_depth': 5, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 150}",-15.355351,-14.600225,-14.243315,-14.849460,-14.110471,-14.631765,0.446005,178,-12.391395,-12.434611,-12.544232,-12.460639,-12.707302,-12.507636,0.111588,14.631765,12.507636
2,0.425218,0.007107,0.008723,0.002244,5,1,200,"{'randomforestregressor__max_depth': 5, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 200}",-15.434430,-14.606842,-14.273765,-14.916145,-14.080456,-14.662328,0.480250,187,-12.367605,-12.446224,-12.573711,-12.451585,-12.714318,-12.510689,0.121292,14.662328,12.510689
3,0.531151,0.011992,0.009561,0.001290,5,1,250,"{'randomforestregressor__max_depth': 5, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 250}",-15.411889,-14.596258,-14.264057,-14.946944,-14.046452,-14.653120,0.486768,184,-12.382046,-12.446415,-12.563267,-12.446650,-12.679294,-12.503535,0.105563,14.653120,12.503535
4,0.613622,0.018099,0.014497,0.005674,5,1,300,"{'randomforestregressor__max_depth': 5, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 300}",-15.392709,-14.583415,-14.236558,-14.924899,-14.079132,-14.643343,0.475021,181,-12.350053,-12.441600,-12.570642,-12.441286,-12.682711,-12.497258,0.116343,14.643343,12.497258


The graph below the the learning curve for hyperparameter of `n_estimators`, which shows the training loss and validation loss changing as training progress:

In [282]:
n_est_df = (
    results.groupby("param_randomforestregressor__n_estimators")
    .agg({"mae_train": "first", "mae_val": "first"})
    .reset_index()
)

fig1 = px.line(
    n_est_df,
    x="param_randomforestregressor__n_estimators",
    y=["mae_train", "mae_val"],
    markers=True,
    title="Effect of n_estimators on MAE",
    labels={"param_randomforestregressor__n_estimators": "n_estimators"},
)

fig1.show()

# fig1.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/n_estimators.html"
# )

The graph below the the learning curve for hyperparameter of `max_depth`, which shows the training loss and validation loss changing as training progress:

In [283]:
max_depth_df = (
    results.groupby("param_randomforestregressor__max_depth")
    .agg({"mae_train": "first", "mae_val": "first"})
    .reset_index()
)

fig1 = px.line(
    max_depth_df,
    x="param_randomforestregressor__max_depth",
    y=["mae_train", "mae_val"],
    markers=True,
    title="Effect of max_depth on MAE",
    labels={"param_randomforestregressor__max_depth": "max_depth"},
)

fig1.show()

# fig1.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/max_depth.html"
# )

The graph below the the learning curve for hyperparameter of `min_samples_leaf`, which shows the training loss and validation loss changing as training progress:

In [284]:
leaf_df = (
    results.groupby("param_randomforestregressor__min_samples_leaf")
    .agg({"mae_train": "first", "mae_val": "first"})
    .reset_index()
)

fig1 = px.line(
    leaf_df,
    x="param_randomforestregressor__min_samples_leaf",
    y=["mae_train", "mae_val"],
    markers=True,
    title="Effect of min_samples_leaf on MAE",
    labels={
        "param_randomforestregressor__min_samples_leaf": "min_samples_leaf"
    },
)

fig1.show()

# fig1.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/min_samples_leaf.html"
# )

The model performance can be visualized below, which the scatter plot shows how predict value is far away from the actual value by the y=x reference line:

In [285]:
y_pred = best_model.predict(X_test)
plot_df = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
fig = px.scatter(
    plot_df,
    x="Actual",
    y="Predicted",
    title="Predicted vs Actual Outage Duration",
    labels={
        "Actual": "Actual Outage Duration",
        "Predicted": "Predicted Outage Duration",
    },
)
max_val = max(plot_df["Actual"].max(), plot_df["Predicted"].max())

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val + 5,
    y1=max_val + 5,
    line=dict(color="red", dash="dash"),
)

fig.update_xaxes(range=[0, max_val + 5])
fig.update_yaxes(range=[0, max_val + 5])

fig.show()
# fig.write_html("/Users/wuyuming/Desktop/proj_4/proj04_page/graph/final.html")

## Step 8: Fairness Analysis

In this section, we want to use permutation test to analysis the fairness of our final model, which our model can performs differently across groups. Doing this analysis intended to prevent the model from systematically favoring one group over another or producing unequal prediction across groups.

We choose the similar grouping approach in the Hypothesis Testing, which we try to compare the model fairness between state have more than 50000$ per capita real GSP and state have less than 50000$ per capita real GSP.

We decided on these groups because we want to check the fairness that model can predict between state have more financial power and state have less financial power. We want to make sure that the model would perform same between two groups mentioned above.

Our evaluation metric will be R square score, which can shows the model's performance on explaining the variability of the actual value.

**Question:** Does our final model performance fairly on predicting `Outage_duratino` between states have more financial power and states less financial power?

**Null Hypothesis:**  The model is fiar. The R squared score is similar between states have more than 50000$ of `Pc_realgsp_state ($)` and states have less than 50000$ of `Pc_realgsp_state ($)`.

**Alternate Hypothesis:** The model is unfiar. The R squared score is different between states have more than 50000$ of `Pc_realgsp_state ($)` and states have less than 50000$ of `Pc_realgsp_state ($)`.

**Test Statistic:** Absolute difference of R squared score




In [287]:
analysis_df = no_outliers.copy()
analysis_df["predict"] = best_model.predict(no_outliers)

analysis_df["realgsp_1/0"] = analysis_df["Pc_realgsp_state ($)"] >= 50000


def fairness_stat(df):
    new_df = df.groupby("realgsp_1/0").apply(
        lambda g: r2_score(g["Outage_duration"], g["predict"])
    )
    return np.abs(new_df.loc[True] - new_df.loc[False])


def fairness_permutation_test(df, state_col, n_perm=1000):

    obs = fairness_stat(df)

    stats = []
    for _ in range(n_perm):
        shuffled = df.copy()
        shuffled[state_col] = np.random.permutation(df[state_col])

        stat = fairness_stat(shuffled)
        stats.append(stat)

    p_value = np.mean(np.array(stats) >= obs)

    return obs, stats, p_value


obs, stats, p_val = fairness_permutation_test(
    analysis_df, "realgsp_1/0", n_perm=1000
)

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/2101747510.py:8: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/2101747510.py:8: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/_h/6x5r9mdd0j5fcx_dg0yzyh9w0000gn/T/ipykernel_84433/2101747510.py:8: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping col

The graph below shows the distribution of permutation test on states have more than 50000$ of real GSP and states have less than 50000$ of real GSP. The significance level is 0.05:

In [288]:
fig = px.histogram(
    stats,
    nbins=40,
    title=f"Permutation Test (p = {p_val:.4f})",
    labels={"value": "Test Statistic"},
)

fig.add_vline(
    x=obs,
    line_color="red",
    line_width=3,
    annotation_text="Observed",
)

fig.add_vline(
    x=np.percentile(stats, 95),
    line_color="black",
    line_width=3,
    annotation_text="significance value = 0.05",
)

fig.update_layout(showlegend=False)

fig.show()
# fig.write_html(
#     "/Users/wuyuming/Desktop/proj_4/proj04_page/graph/fairness.html"
# )

Through the graph above, we can see the observed absolute difference of R squared score has a p-value 0.7620. The result show we fail to reject the null hypothesis. This result means we not have sufficient evidence ot conclude that the model perfoms differently across the two groups. We do not find sufficient evidence to confirm the model performed unfairly.